# ML-P-8 — Random Forest vs XGBoost
Задания 8.1 и 8.2. Начните с `pip install -r requirements.txt`.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("o_churn_synthetic.csv")
X = df.drop(columns=["churn", "subscriber_code"])
y = df["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

def make_prep():
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["region", "tariff"])
    ], remainder="passthrough")

print("train:", X_train.shape, "test:", X_test.shape)
print("churn:", y.value_counts().to_dict())

In [ ]:
import time
from sklearn.metrics import f1_score, roc_auc_score

results = []
def fit_and_report(name, pipe):
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    sec = time.perf_counter() - t0
    f1_train = f1_score(y_train, pipe.predict(X_train))
    f1_test = f1_score(y_test, pipe.predict(X_test))
    auc_test = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    results.append({"Модель": name, "F1 train": f1_train, "F1 test": f1_test,
                    "ROC-AUC test": auc_test, "Время, с": sec,
                    "Разрыв F1": f1_train - f1_test})
    print(f"{name:15s} F1 train={f1_train:.3f} F1 test={f1_test:.3f} "
          f"ROC-AUC test={auc_test:.3f} {sec:.2f} c")
    return pipe

In [ ]:
from xgboost import XGBClassifier

xgb = make_pipeline(make_prep(), XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    random_state=42, n_jobs=1, eval_metric="logloss"
))
fit_and_report("XGBoost", xgb)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = make_pipeline(make_prep(), RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=1
))
fit_and_report("Random Forest", rf)

In [ ]:
comparison = pd.DataFrame(results).set_index("Модель")
display(comparison.round(3))
print("Лучший F1 test:", comparison["F1 test"].idxmax())
print("Лучший ROC-AUC:", comparison["ROC-AUC test"].idxmax())
print("Большой разрыв train/test означает риск переобучения.")

In [ ]:
import matplotlib.pyplot as plt
ax = comparison[["F1 train", "F1 test", "ROC-AUC test"]].plot.bar(
    figsize=(9, 5), rot=0, title="Random Forest vs XGBoost")
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
plt.tight_layout()
plt.show()